# Hamburg Fluctuating Pain Database quickstart

This notebook shows a minimal read-only workflow for the released DuckDB database:

- connect to the database,
- inspect the available tables,
- load a small participant-level slice from the merged feature table,
- access questionnaire data, and
- run one lightweight processing utility on raw stimulus data.


In [27]:
from pathlib import Path
import os
import sys

import polars as pl

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if not (repo_root / "src").exists():
    raise FileNotFoundError(
        "Open this notebook from the repository root or the notebooks directory."
    )

os.chdir(repo_root)
if repo_root.as_posix() not in sys.path:
    sys.path.insert(0, repo_root.as_posix())

from src.data.database_manager import DatabaseManager
from src.data.data_processing import create_feature_data_df

pl.Config.set_tbl_rows(8)
pl.Config.set_tbl_cols(10)

db = DatabaseManager()

## 1. Inspect the released tables

 The most elegant way to connect to the database is via a context manager (`with` statement), which automatically handles connection opening and closing.

In [28]:
with db:
    tables = db.execute("SHOW TABLES").pl().sort("name")
tables

name
str
"""Calibration_Results"""
"""Explore_Data"""
"""Explore_EDA"""
"""Explore_Face"""
…
"""Raw_Pupil"""
"""Raw_Stimulus"""
"""Seeds"""
"""Trials_Info"""


## 2. Summarize a few core tables

The merged `Feature_Data` table is usually the easiest place to start for cross-modality analyses.


In [29]:
with db:
    summary = pl.concat(
        [
            db.execute(
                "SELECT 'Feature_Data' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT participant_id) AS participants FROM Feature_Data"
            ).pl(),
            db.execute(
                "SELECT 'Feature_EEG' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT participant_id) AS participants FROM Feature_EEG"
            ).pl(),
            db.execute(
                "SELECT 'Raw_Stimulus' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT participant_id) AS participants FROM Raw_Stimulus"
            ).pl(),
            db.execute(
                "SELECT 'Questionnaire_PANAS' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT participant_id) AS participants FROM Questionnaire_PANAS"
            ).pl(),
        ],
        how="vertical",
    ).sort("table_name")
summary

table_name,rows,participants
str,i64,i64
"""Feature_Data""",929316,45
"""Feature_EEG""",32279055,45
"""Questionnaire_PANAS""",90,45
"""Raw_Stimulus""",965275,45


## 3. Load a participant-level slice from `Feature_Data`

The `exclude_problematic` flag removes trials with modality-specific measurement issues. Here we preview the first trial for participant 1.


In [30]:
with db:
    feature_data_p1 = db.get_trials(
        "Feature_Data",
        exclude_problematic=True,
        participant_ids=[1],
    )

feature_data_p1.filter(pl.col("trial_number") == 1).select(
    [
        "participant_id",
        "trial_number",
        "normalized_timestamp",
        "temperature",
        "rating",
        "heart_rate",
        "pupil",
        "mouth_open",
    ]
).head(10)

participant_id,trial_number,normalized_timestamp,temperature,rating,heart_rate,pupil,mouth_open
u8,u8,f64,f64,f64,f64,f64,f64
1,1,0.0,0.0,0.38375,75.629597,4.608068,0.004032
1,1,100.0,0.000056,0.395091,76.071842,4.607972,0.003763
1,1,200.0,0.000241,0.406316,76.571757,4.610181,0.003514
1,1,300.0,0.000572,0.415328,76.593026,4.614996,0.003353
…,…,…,…,…,…,…,…
1,1,600.0,0.002451,0.444389,76.524291,4.62753,0.00288
1,1,700.0,0.003366,0.45839,76.719166,4.637487,0.002731
1,1,800.0,0.004432,0.475577,78.274291,4.646029,0.00256
1,1,900.0,0.005647,0.487499,80.239936,4.645539,0.002402


## 4. Check the merged feature schema

DuckDB's `DESCRIBE` output is a quick way to inspect available columns and types without loading the full table into memory.


In [31]:
with db:
    db.execute("DESCRIBE Feature_Data").pl()

## 5. Access questionnaire data

Questionnaire tables can be retrieved directly with `get_table` because they are not time-series tables.


In [32]:
with db:
    df = (
        db.get_table("Questionnaire_PANAS")
        .select(
            [
                "participant_id",
                "age",
                "gender",
                "positive_affect",
                "negative_affect",
            ]
        )
        .head(8)
    )
df

participant_id,age,gender,positive_affect,negative_affect
u8,i64,str,f64,f64
1,22,"""m""",3.5,1.2
1,22,"""m""",3.5,1.0
2,23,"""m""",3.9,1.3
2,23,"""m""",3.8,1.1
3,23,"""m""",1.9,1.2
3,23,"""m""",1.8,1.0
4,35,"""f""",2.8,1.2
4,35,"""f""",2.3,1.3


## 6. Run one lightweight processing utility on raw stimulus data

The repository also includes reusable transformation functions. Here we take a short raw stimulus slice and convert it with `create_feature_data_df`.


In [33]:
with db:
    raw_stimulus = (
        db.get_trials("Raw_Stimulus", exclude_problematic=False, participant_ids=[1])
        .filter(pl.col("trial_number") == 1)
        .head(8)
    )
raw_stimulus

trial_id,trial_number,participant_id,rownumber,timestamp,temperature,rating
u16,u8,u8,u32,f64,f64,f64
1,1,1,0,200204.8709,45.5,38.375
1,1,1,1,200305.472,45.500118,39.875
1,1,1,2,200405.205,45.500472,40.875
1,1,1,3,200505.9354,45.501061,41.75
1,1,1,4,200606.2101,45.501886,42.625
1,1,1,5,200705.9479,45.502946,43.875
1,1,1,6,200805.6808,45.504241,44.625
1,1,1,7,200906.4115,45.505771,46.25


In [34]:
create_feature_data_df("Feature_Stimulus", raw_stimulus)

trial_id,trial_number,participant_id,timestamp,temperature,rating
u16,u8,u8,f64,f64,f64
1,1,1,200204.8709,0.0,0.38375
1,1,1,200305.472,0.020433,0.39875
1,1,1,200405.205,0.081727,0.40875
1,1,1,200505.9354,0.183862,0.4175
1,1,1,200606.2101,0.326807,0.42625
1,1,1,200705.9479,0.510519,0.43875
1,1,1,200805.6808,0.734939,0.44625
1,1,1,200906.4115,1.0,0.4625
